<a href="https://colab.research.google.com/github/busybee-123/Pollinator_Cam/blob/main/Train_Yolo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Label all images

In [ ]:
!pip install -q transformers accelerate

import os
import torch
from PIL import Image
from transformers import OwlViTProcessor, OwlViTForObjectDetection
from google.colab import drive

drive.mount('/content/drive')

# 1. Load the Zero-Shot AI Model
processor = OwlViTProcessor.from_pretrained("google/owlvit-base-patch32")
model = OwlViTForObjectDetection.from_pretrained("google/owlvit-base-patch32").to("cuda")

image_dir = "/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/bee_dataset/images"
label_dir = "/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/bee_dataset/labels"
os.makedirs(label_dir, exist_ok=True)

class_mapping = {
    "nomada flavoguttata": 0,
    "bombus terrestris": 1,
    "megachile ligniseca": 2,
    "andrena flavipes": 3,
    "melitta haemorrhoidalis": 4,
    "andrena clarkella": 5,
    "colletes hederae": 6,
    "chelostoma campanularum": 7,
    "andrena fulva": 8,
    "lasioglossum smeathmanellum": 9,
    "andrena dorsata": 10,
    "nomada fulvicornis": 11,
    "andrena ovatula": 12,
    "halictus tumulorum": 13,
    "megachile willughbiella": 14,
    "sphecodes monilicornis": 15,
    "bombus lapidarius": 16,
    "andrena gravida": 17,
    "bombus ruderatus": 18,
    "ceratina cyanea": 19
}

# 2. Process the images
for filename in os.listdir(image_dir):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        img_path = os.path.join(image_dir, filename)
        image = Image.open(img_path).convert("RGB")
        W, H = image.size

        # Determine class from filename
        class_id = next((cid for kw, cid in class_mapping.items() if kw in filename.lower()), None)
        if class_id is None: continue

        # Ask the AI to find "a bee"
        inputs = processor(text=[["a bee"]], images=image, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = model(**inputs)

        # Target sizes are required to convert normalized predictions to actual box coordinates
        target_sizes = torch.Tensor([[H, W]]).to("cuda")
        results = processor.post_process_object_detection(outputs=outputs, target_sizes=target_sizes, threshold=0.25)

        # Extract the highest confidence box found
        boxes, scores = results[0]["boxes"], results[0]["scores"]
        if len(boxes) > 0:
            best_idx = torch.argmax(scores).item()
            box = boxes[best_idx].cpu().numpy() # [xmin, ymin, xmax, ymax]

            # Convert [xmin, ymin, xmax, ymax] to YOLO format [x_center, y_center, width, height] (normalized)
            x_center = ((box[0] + box[2]) / 2) / W
            y_center = ((box[1] + box[3]) / 2) / H
            width = (box[2] - box[0]) / W
            height = (box[3] - box[1]) / H

            # Write out the label file
            label_filename = os.path.splitext(filename)[0] + ".txt"
            with open(os.path.join(label_dir, label_filename), "w") as f:
                f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

print("AI auto-labeling finished successfully!")

ValueError: Mountpoint must not already contain files

In [ ]:
# 2. Process the images
for filename in os.listdir(image_dir):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        img_path = os.path.join(image_dir, filename)
        image = Image.open(img_path).convert("RGB")
        W, H = image.size

        # Determine class from filename
        class_id = None
        # Iterate through class_mapping to find a match in the filename
        # Convert class_mapping keys to match the format in filenames (spaces instead of underscores)
        for class_name_snake_case, cid in class_mapping.items():
            class_name_in_filename_format = class_name_snake_case.replace('_', ' ')
            if class_name_in_filename_format in filename.lower():
                class_id = cid
                break # Found a match, assign class_id and exit the loop
        if class_id is None:
            # If no species name from class_mapping is found, skip this file
            print(f"Warning: No class found for filename: {filename}. Skipping.")
            continue

        # Ask the AI to find "a bee"
        inputs = processor(text=[["a bee"]], images=image, return_tensors="pt", do_rescale=False).to("cuda")
        with torch.no_grad():
            outputs = model(**inputs)

        # Target sizes are required to convert normalized predictions to actual box coordinates
        target_sizes = torch.Tensor([[H, W]]).to("cuda")
        # Lowering the threshold to capture more detections, even if less confident.
        results = processor.image_processor.post_process_object_detection(outputs=outputs, target_sizes=target_sizes, threshold=0.01)

        # Extract the highest confidence box found
        boxes, scores = results[0]["boxes"], results[0]["scores"]
        if len(boxes) > 0:
            best_idx = torch.argmax(scores).item()
            box = boxes[best_idx].cpu().numpy() # [xmin, ymin, xmax, ymax]

            # Convert [xmin, ymin, xmax, ymax] to YOLO format [x_center, y_center, width, height] (normalized)
            x_center = ((box[0] + box[2]) / 2) / W
            y_center = ((box[1] + box[3]) / 2) / H
            width = (box[2] - box[0]) / W
            height = (box[3] - box[1]) / H

            # Write out the label file
            label_filename = os.path.splitext(filename)[0] + ".txt"
            with open(os.path.join(label_dir, label_filename), "w") as f:
                f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

print("AI auto-labeling finished successfully!")

AI auto-labeling finished successfully!


In [ ]:
import os
import shutil
import random
from sklearn.model_selection import train_test_split

# 1. Define your current Google Drive paths
drive_images_dir = '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/bee_dataset/images'
drive_labels_dir = '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/bee_dataset/labels'

# 2. Define the output local directory structure
base_out = '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/draft_dataset'
os.makedirs(f"{base_out}/images/train", exist_ok=True)
os.makedirs(f"{base_out}/images/val", exist_ok=True)
os.makedirs(f"{base_out}/labels/train", exist_ok=True)
os.makedirs(f"{base_out}/labels/val", exist_ok=True)

# Gather all unique image basenames (assumes matching names like bee1.jpg and bee1.txt)
image_extensions = ('.jpg', '.jpeg', '.png')
all_images = [f for f in os.listdir(drive_images_dir) if f.lower().endswith(image_extensions)]
basenames = [os.path.splitext(f)[0] for f in all_images]

# 3. Split into Train (80%) and Validation (20%)
train_names, val_names = train_test_split(basenames, test_size=0.2, random_state=42)

def move_files(name_list, split_type):
    for name in name_list:
        # Find matching image extension
        img_name = next((f for f in all_images if os.path.splitext(f)[0] == name), None)
        if img_name:
            # Move Image
            shutil.copy(os.path.join(drive_images_dir, img_name), f"{base_out}/images/{split_type}/{img_name}")
            # Move Label
            lbl_name = f"{name}.txt"
            if os.path.exists(os.path.join(drive_labels_dir, lbl_name)):
                shutil.copy(os.path.join(drive_labels_dir, lbl_name), f"{base_out}/labels/{split_type}/{lbl_name}")

move_files(train_names, 'train')
move_files(val_names, 'val')
print(f"Data split successfully! Training: {len(train_names)} | Validation: {len(val_names)}")

ValueError: Mountpoint must not already contain files

In [ ]:
import yaml
import os

# Define paths based on your previous cells
base_out = '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/draft_dataset'
train_images_path = f'{base_out}/images/train'
val_images_path = f'{base_out}/images/val'

# Retrieve class_mapping from the previous cell's context
# (Assuming class_mapping is available in the global scope from cell si2Y3eVbOw1Z)
# If not, you might need to re-define or re-run the cell where it's defined.
class_mapping = {
    "nomada flavoguttata": 0,
    "bombus terrestris": 1,
    "megachile ligniseca": 2,
    "andrena flavipes": 3,
    "melitta haemorrhoidalis": 4,
    "andrena clarkella": 5,
    "colletes hederae": 6,
    "chelostoma campanularum": 7,
    "andrena fulva": 8,
    "lasioglossum smeathmanellum": 9,
    "andrena dorsata": 10,
    "nomada fulvicornis": 11,
    "andrena ovatula": 12,
    "halictus tumulorum": 13,
    "megachile willughbiella": 14,
    "sphecodes monilicornis": 15,
    "bombus lapidarius": 16,
    "andrena gravida": 17,
    "bombus ruderatus": 18,
    "ceratina cyanea": 19
}

# Get class names sorted by their integer IDs
class_names = [name for name, _ in sorted(class_mapping.items(), key=lambda item: item[1])]

# Create the data.yaml content
data = {
    'train': train_images_path,  # Path to training images
    'val': val_images_path,      # Path to validation images
    'nc': len(class_names),      # Number of classes
    'names': class_names         # Class names
}

# Define the output path for data.yaml
data_yaml_path = '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/bee_dataset/data2.yaml'

# Ensure the directory for data.yaml exists
os.makedirs(os.path.dirname(data_yaml_path), exist_ok=True)

# Write the data to the YAML file
with open(data_yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print(f"Created data.yaml at: {data_yaml_path}")

Created data.yaml at: /content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/bee_dataset/data2.yaml


# Does not work - Use "train yolo 2"

In [ ]:
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -r requirements.txt



Cloning into 'yolov5'...
remote: Enumerating objects: 18370, done.
remote: Counting objects: 100% (67/67), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 18370 (delta 43), reused 23 (delta 23), pack-reused 18303 (from 2)
Receiving objects: 100% (18370/18370), 17.52 MiB | 19.60 MiB/s, done.
Resolving deltas: 100% (12486/12486), done.
/content/yolov5
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 10.1 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [ ]:
!python train.py --weights yolov5s.pt --data '/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/draft_dataset/data2.yaml' --freeze 24

wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: (30 second timeout) 3
wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
train: weights=yolov5s.pt, cfg=, data=/content/drive/MyDrive/Colab Notebooks/Pollinator Camera Project/draft_dataset/data2.yaml, hyp=data/hyps/hyp.scratch-low.yaml, epochs=100, batch_size=16, imgsz=640, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, evolve_population=data/hyps, resume_evolve=None, bucket=, cache=None, image_weights=False, device=, multi_scale=False, single_cls=False, optimizer=SGD, sync_bn=False, workers=8, project=runs/train, name=exp, exist_ok=Fa